# Multimodal Meme Classifier — Demo

This notebook is a visual entry point to the refactored code.
It does NOT retrain models (no trained checkpoints are shipped).

**Three sections:**
1. **Dataset exploration** — what the data looks like, per-class counts, caption stats.
2. **Model walkthrough** — build every architecture through `src.models.factory.build_model` and confirm the forward pass works.
3. **Paper results** — figures extracted from the original IEEE 2024 paper.

> Prerequisites:
>   - `pip install -r requirements.txt`
>   - (for §1 only) `data/top5_memes_tidy.tsv` from the Kaggle dataset preprocessed to 5 labels.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

sns.set_theme(style='whitegrid')
print(f'Repo root: {REPO_ROOT}')
print(f'TensorFlow: {tf.__version__}')


## §1 Dataset Exploration

The paper uses a preprocessed TSV at `data/top5_memes_tidy.tsv`, derived from the
[ImgFlip-Scraped Memes Caption Dataset](https://www.kaggle.com/datasets/abhishtagatya/imgflipscraped-memes-caption-dataset?select=memes_data.tsv) on Kaggle.
The five classes used for multiclass classification:
Who Killed Hannibal, Scared Cat, Sleeping Shaq, Uncle Sam, Peter Parker Cry.
Binary classification uses only the first two.

In [ ]:
DATA_PATH = REPO_ROOT / 'data' / 'top5_memes_tidy.tsv'

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH, sep='\t')
    print(f'{len(df)} rows, columns: {list(df.columns)}')
    df.head(3)
else:
    print(f'[skipping §1] — {DATA_PATH} not found.')
    print('Place the preprocessed TSV in data/ to re-run the exploration cells.')
    df = None


In [ ]:
if df is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    counts = df['MemeLabel'].value_counts()
    counts.plot.bar(ax=ax, color=sns.color_palette('viridis', len(counts)))
    ax.set_ylabel('# captions')
    ax.set_title('Class distribution (top-5 memes)')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
if df is not None:
    df = df.copy()
    df['caption_len'] = df['CaptionText'].astype(str).str.split().str.len()
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(data=df, x='caption_len', hue='MemeLabel', bins=30,
                 element='step', stat='density', common_norm=False, ax=ax)
    ax.set_xlabel('caption length (# tokens)')
    ax.set_title('Caption length distribution by class')
    plt.tight_layout()
    plt.show()


## §2 Model Walkthrough

All four architectures are built via `build_model(cfg)`. Each YAML config
in `configs/` is a valid input to this one function.

In [ ]:
from src.config import load_config
from src.models.factory import build_model

CONFIGS = {
    'Text (BERT, multiclass)':   'configs/text/bert_multi.yaml',
    'Image (VGG16, multiclass)': 'configs/image/vgg_multi.yaml',
    'Early fusion (multiclass)': 'configs/fusion/early_multi.yaml',
    'Late fusion (multiclass)':  'configs/fusion/late_multi.yaml',
}

models = {}
for label, cfg_path in CONFIGS.items():
    cfg = load_config(REPO_ROOT / cfg_path)
    models[label] = build_model(cfg['model'])
    print(f'{label}: {models[label].count_params():,} parameters')


In [ ]:
# Summary of the flagship model
models['Late fusion (multiclass)'].summary()


In [ ]:
# Forward-pass sanity check on dummy inputs
BATCH = 2
dummy_text_ids = tf.ones([BATCH, 128], dtype=tf.int32)
dummy_text_mask = tf.ones([BATCH, 128], dtype=tf.int32)
dummy_image = tf.random.uniform([BATCH, 224, 224, 3])

inputs_by_model = {
    'Text (BERT, multiclass)':   [dummy_text_ids, dummy_text_mask],
    'Image (VGG16, multiclass)': dummy_image,
    'Early fusion (multiclass)': [dummy_text_ids, dummy_text_mask, dummy_image],
    'Late fusion (multiclass)':  [dummy_text_ids, dummy_text_mask, dummy_image],
}

for label, model in models.items():
    out = model(inputs_by_model[label], training=False)
    print(f'{label:40s}→ output shape {tuple(out.shape)}')


## §3 Paper Results

Figures below are extracted from the notebooks that produced the original
IEEE paper. The core numerical comparison is in
[assets/paper_results/main_table.md](../assets/paper_results/main_table.md).

Key paper findings:
* Early fusion converges faster than late fusion on the multiclass task.
* Binary models reach high accuracy within the first epoch; multiclass takes longer.
* Late fusion averages independent predictions, so it misses lower-level text/image interactions.

In [ ]:
from IPython.display import Image, display

featured = [
    ('early_fusion_multi__fig03.png', 'Early fusion (multiclass) — training curve'),
    ('late_fusion_multi__fig03.png',  'Late fusion (multiclass) — training curve'),
    ('early_fusion_binary__fig03.png', 'Early fusion (binary) — training curve'),
    ('late_fusion_binary__fig03.png',  'Late fusion (binary) — training curve'),
]
for fname, caption in featured:
    path = REPO_ROOT / 'assets' / 'paper_results' / fname
    if path.exists():
        print(caption)
        display(Image(filename=str(path)))
